# RAG avanzado — Asistente de derecho laboral

SI4006 · Equipo Lawten

Técnicas de recuperación elegidas a partir de dónde falla el RAG ingenuo: búsqueda híbrida (BM25 + denso con RRF),
reranking con el cross-encoder de M1 y transformación de la consulta (multi-query con términos jurídicos). Mismo corpus,
mismo decoder, misma plantilla de respuesta, mismo top-k, mismo eval set y mismo juez: lo único que cambia entre sistemas
es cómo se recuperan y ordenan las tres referencias, así que cualquier diferencia es atribuible a la técnica.

Salidas: `scorecard_rag_avanzado.csv` y `scorecard_rag_avanzado_detalle.csv` (sistema avanzado) y
`ablacion_rag_avanzado.csv` y `ablacion_rag_avanzado_detalle.csv` (los cinco sistemas).

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as version_instalada
from pathlib import Path

from packaging.version import Version

REPO_URL = 'https://github.com/Bosnape/cabrejos-ortiz-valencia-lopez.git'
REPO_BRANCH = 'main'
NOTEBOOK_REL = Path('notebooks/rag/rag_avanzado.ipynb')

try:
    import google.colab  # type: ignore
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def buscar_raiz(inicio):
    inicio = Path(inicio).resolve()
    return next((p for p in (inicio, *inicio.parents) if (p / NOTEBOOK_REL).is_file()), None)

RAIZ = buscar_raiz(Path.cwd())
if RAIZ is None and EN_COLAB:
    destino = Path('/content/cabrejos-ortiz-valencia-lopez')
    if not destino.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(destino)], check=True)
    RAIZ = destino
if RAIZ is None:
    raise RuntimeError('Abra el notebook dentro del repositorio clonado.')
os.chdir(RAIZ)

if EN_COLAB:
    # PEFT rechaza el torchao preinstalado en algunos runtimes; aqui se usa LoRA convencional.
    try:
        torchao = version_instalada('torchao')
    except PackageNotFoundError:
        torchao = None
    if torchao is not None and Version(torchao) < Version('0.16.0'):
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    # Solo lo que este notebook importa; pillow se fija para que ningun paquete lo actualice a mitad de sesion.
    pillow_fija = version_instalada('pillow')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sentence-transformers==6.0.1',
                    'transformers==5.16.1',
                    'anthropic==1.4.0',
                    'pandas==3.0.5',
                    'python-dotenv==1.2.3',
                    'peft==0.20.0',
                    'accelerate==1.14.0',
                    'chromadb',
                    'rank_bm25', f'pillow=={pillow_fija}'], check=True)

requeridos = [
    Path('notebooks/evaluation/rubrica.json'),
    Path('notebooks/evaluation/eval_set.json'),
    Path('notebooks/evaluation/harness.py'),
    Path('data/diccionario_articulos.csv'),
    Path('notebooks/evaluation/scorecard_baseline.csv'),
    Path('notebooks/rag/resultados/ingenuo/scorecard_rag_ingenuo_detalle.csv'),
    Path('s04_lora_adapter/adapter_config.json'),
    Path('s04_lora_adapter/adapter_model.safetensors'),
]
faltantes = [str(r) for r in requeridos if not (RAIZ / r).is_file()]
if faltantes:
    raise FileNotFoundError('Faltan archivos: ' + ', '.join(faltantes))
sys.path.insert(0, str(RAIZ / 'notebooks/evaluation'))
print('Raiz del proyecto:', RAIZ)

In [ ]:
import json
import random
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import torch
from IPython.display import display

import harness as h

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
GEN_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
JUEZ_MODEL = 'claude-haiku-4-5'
MODELO_CE = 'dccuchile/bert-base-spanish-wwm-cased'
RUTA_ADAPTADOR = 's04_lora_adapter'
N_CAND = 20               # candidatos que aporta cada buscador antes de fusionar o reordenar
K_RRF = 60
N_REFORMULACIONES = 3
SISTEMA_AVANZADO = 'C2_hibrida_rerank'   # sistema que se exporta como scorecard_rag_avanzado.csv
TOP_K = h.TOP_K
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

## 1. Corpus

Los 143 artículos del diccionario, cada uno como un solo fragmento. `textos_ce` lleva el prefijo de la fuente porque así
recibe el texto el cross-encoder de M1.

In [ ]:
eval_set = h.cargar_eval_set(RAIZ)
corpus = h.cargar_corpus(RAIZ)
assert h.validar_eval_set(eval_set, corpus)
print(f'{len(eval_set)} casos; {sum(e["is_adversarial"] for e in eval_set)} adversariales; {len(corpus)} artículos en el corpus.')
display(pd.DataFrame(eval_set).assign(input=lambda d: d['input'].str.slice(0, 90) + '...')[['id', 'case_type', 'is_adversarial', 'input']])
textos_ce = (corpus['fuente'] + '. ' + corpus['texto']).tolist()

## 2. Buscadores base

- **Denso**: embeddings multilingües y un índice Chroma con distancia coseno.
- **BM25**: coincidencia léxica. Se normalizan acentos y puntuación antes de tokenizar, para que "salario," y "Salario"
  cuenten como la misma palabra.
- **Cross-encoder de M1** (BETO con adaptador LoRA): puntúa el par (consulta, artículo) leyéndolos juntos. Es el modelo del
  baseline; aquí actúa como reranker sobre una lista corta de candidatos.

Cada buscador devuelve una lista `(índice, puntaje)` ordenada de mayor a menor.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer(EMBED_MODEL)

cliente_vectorial = chromadb.Client()
try:
    cliente_vectorial.delete_collection('rag_avanzado')
except Exception:
    pass
coleccion = cliente_vectorial.create_collection('rag_avanzado', metadata={'hnsw:space': 'cosine'})
coleccion.add(ids=[str(i) for i in range(len(corpus))], documents=corpus['texto'].tolist(),
              embeddings=modelo_embeddings.encode(corpus['texto'].tolist(), show_progress_bar=False).tolist())

def buscar_densa(consulta, n=N_CAND):
    r = coleccion.query(query_embeddings=modelo_embeddings.encode([consulta]).tolist(), n_results=n)
    return [(int(i), 1 - d) for i, d in zip(r['ids'][0], r['distances'][0])]

print('Artículos indexados en Chroma:', coleccion.count())

In [ ]:
from rank_bm25 import BM25Okapi

def tokenizar(texto):
    texto = unicodedata.normalize('NFD', texto.lower())
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return re.findall(r'\w+', texto)

bm25 = BM25Okapi([tokenizar(t) for t in corpus['texto']])

def buscar_bm25(consulta, n=N_CAND):
    scores = bm25.get_scores(tokenizar(consulta))
    orden = np.argsort(scores)[::-1][:n]
    return [(int(i), float(scores[i])) for i in orden]

print('BM25 listo.')

In [ ]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ce_tok = AutoTokenizer.from_pretrained(MODELO_CE)
ce_base = AutoModelForSequenceClassification.from_pretrained(MODELO_CE, num_labels=2)
modelo_ce = PeftModel.from_pretrained(ce_base, RAIZ / RUTA_ADAPTADOR).to(device).eval()

def puntuar_ce(consulta, candidatos):
    candidatos = list(candidatos)
    entradas = ce_tok([consulta] * len(candidatos), [textos_ce[i] for i in candidatos],
                      truncation='only_second', max_length=512, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = modelo_ce(**entradas).logits
    puntajes = torch.softmax(logits, dim=-1)[:, 1].cpu().tolist()
    return sorted(zip(candidatos, puntajes), key=lambda x: -x[1])

print('Cross-encoder de M1 (LoRA) listo.')

## 3. Fusión

RRF (Reciprocal Rank Fusion) combina listas por **posición**: cada artículo suma `1 / (k + puesto)` en cada lista. Los
puntajes de BM25 y del coseno tienen escalas incomparables, y RRF no depende de ellas.

In [ ]:
def fusionar_rrf(rankings, k_rrf=K_RRF, top=None):
    puntos = {}
    for ranking in rankings:
        for puesto, (idx, _) in enumerate(ranking):
            puntos[idx] = puntos.get(idx, 0) + 1.0 / (k_rrf + puesto + 1)
    fusion = sorted(puntos.items(), key=lambda x: -x[1])
    return fusion[:top] if top else fusion

print('RRF listo.')

## 4. Generación

El mismo decoder y la misma plantilla de respuesta de todos los sistemas (ver `notebooks/evaluation/harness.py`).
`generar_texto` también sirve para extraer términos jurídicos en el sistema D.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained(GEN_MODEL)
generador = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype='auto').to(device).eval()

def generar_texto(sistema, usuario, max_new_tokens):
    chat = [{'role': 'system', 'content': sistema}, {'role': 'user', 'content': usuario}]
    prompt = tokenizador.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    entrada = tokenizador(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        salida = generador.generate(**entrada, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tokenizador.eos_token_id)
    return tokenizador.decode(salida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True).strip()

print('Decoder listo:', GEN_MODEL)

## 5. Los cinco sistemas

Cada uno devuelve los `TOP_K` índices que verá el decoder.

| Sistema | Qué hace | Qué aísla |
|---|---|---|
| **A_ingenuo** | Denso, top-3 | Línea base de RAG |
| **B_hibrida** | RRF(denso, BM25), top-3 | Aporte de BM25 sobre el denso |
| **C1_denso_rerank** | Denso, 20 candidatos; el cross-encoder de M1 reordena | Aporte del reranker sobre el denso |
| **C2_hibrida_rerank** | RRF(denso, BM25), 20 candidatos; el cross-encoder reordena | Híbrida y reranking juntos |
| **D_multiquery** | C2, pero buscando con la consulta y 3 frases con los términos jurídicos que Qwen extrae del relato, fusionadas con RRF; el cross-encoder reordena con la consulta original | Aporte de traducir el relato a términos jurídicos |

Comparaciones que dan la ablación: B contra A (BM25), C1 contra A (reranker sobre denso), C2 contra B (reranker sobre
híbrida), C2 contra C1 (BM25 con reranker) y D contra C2 (multi-query).

In [ ]:
SISTEMA_REFORMULAR = ('Eres un asistente jurídico laboral colombiano. Identificas los temas legales de una '
                      'consulta. Responde SOLO con frases cortas de búsqueda, una por línea, sin numeración ni explicaciones.')

def reformular(consulta, n=N_REFORMULACIONES):
    texto = generar_texto(
        SISTEMA_REFORMULAR,
        f'Consulta de un trabajador:\n{consulta}\n\nEscribe {n} frases cortas (máximo 8 palabras cada una) con los '
        f'términos jurídicos laborales (instituciones, derechos o figuras legales) que describen el problema. '
        f'No repitas los hechos de la consulta.',
        60)
    variantes = []
    for linea in texto.split('\n'):
        v = re.sub(r'^[\s\-•*\d\.\)]+', '', linea).strip()
        if len(v) > 5 and v not in variantes:
            variantes.append(v)
    return variantes[:n]

def candidatos_hibridos(consulta):
    return fusionar_rrf([buscar_densa(consulta), buscar_bm25(consulta)])

def recuperar_A(consulta):
    return [i for i, _ in buscar_densa(consulta, TOP_K)]

def recuperar_B(consulta):
    return [i for i, _ in candidatos_hibridos(consulta)[:TOP_K]]

def recuperar_C1(consulta):
    candidatos = [i for i, _ in buscar_densa(consulta)]
    return [i for i, _ in puntuar_ce(consulta, candidatos)[:TOP_K]]

def recuperar_C2(consulta):
    candidatos = [i for i, _ in candidatos_hibridos(consulta)[:N_CAND]]
    return [i for i, _ in puntuar_ce(consulta, candidatos)[:TOP_K]]

def recuperar_D(consulta):
    consultas = [consulta] + reformular(consulta)
    recuperar_D.consultas = consultas[1:]
    fusionada = fusionar_rrf([candidatos_hibridos(v) for v in consultas], top=N_CAND)
    return [i for i, _ in puntuar_ce(consulta, [i for i, _ in fusionada])[:TOP_K]]

def construir_sistema(recuperar_fn):
    def sistema(pregunta):
        inicio = time.perf_counter()
        indices = recuperar_fn(pregunta)
        sistema.latencia_recuperacion = time.perf_counter() - inicio
        sistema.consultas_reformuladas = list(getattr(recuperar_fn, 'consultas', []))
        sistema.ultimo_contexto = h.referencias_de(indices, corpus)
        return h.responder(pregunta, sistema.ultimo_contexto, generar_texto)
    return sistema

sistemas = {
    'A_ingenuo': construir_sistema(recuperar_A),
    'B_hibrida': construir_sistema(recuperar_B),
    'C1_denso_rerank': construir_sistema(recuperar_C1),
    'C2_hibrida_rerank': construir_sistema(recuperar_C2),
    'D_multiquery': construir_sistema(recuperar_D),
}
sistema_rag_avanzado = sistemas[SISTEMA_AVANZADO]
print('Sistemas listos:', list(sistemas), '| avanzado:', SISTEMA_AVANZADO)

## 6. Evaluación

Mismo eval set, misma rúbrica y mismo juez del harness (`notebooks/evaluation/harness.py`): recall@3 de la norma
esperada, juez con esquema JSON estricto y mitigación de sesgo de posición (promedio de ambos órdenes), y la misma regla de
acierto. Además de la latencia total por consulta se mide la **latencia de recuperación** por separado, porque la total la
domina la longitud de la respuesta generada.

In [ ]:
rubrica_datos, RUBRICA = h.cargar_rubrica(RAIZ)
RUBRICA_VERSION = rubrica_datos['version']
juez = h.crear_juez(RUBRICA, JUEZ_MODEL, RAIZ, EN_COLAB)
print(f'Rúbrica v{RUBRICA_VERSION}')

### Corrida

Cada sistema se evalúa sobre los 13 casos con una respuesta fresca. Tarda varios minutos.

In [ ]:
METADATOS = {
    'modelo_embeddings': EMBED_MODEL, 'modelo_cross_encoder': MODELO_CE, 'n_candidatos': N_CAND,
    'modelo_generador': GEN_MODEL, 'modelo_juez': JUEZ_MODEL, 'top_k': TOP_K, 'seed': SEED,
    'version_rubrica': RUBRICA_VERSION,
}

resultados_sistemas = {}
for nombre, sistema in sistemas.items():
    print('=' * 20, nombre)
    resultados_sistemas[nombre] = h.harness(eval_set, sistema, juez, nombre=nombre, metadatos=METADATOS)

filas = pd.DataFrame([f for r in resultados_sistemas.values() for f in r['filas']])
detalle = pd.DataFrame([d for r in resultados_sistemas.values() for d in r['detalle']])

## 7. Resultados

### Tabla evolutiva y ablación

La fila `baseline_M1` viene de `scorecard_baseline.csv` (sin RAG); las demás se acaban de medir en esta sesión. La latencia
de recuperación no se reporta para el baseline.

In [ ]:
baseline = pd.read_csv(RAIZ / 'notebooks/evaluation/scorecard_baseline.csv')
columnas_resumen = ['recall_3_medio', 'juez_promedio', 'aciertos', 'latencia_mediana_seg', 'latencia_recuperacion_mediana_seg']
resumen = pd.DataFrame({'baseline_M1': h.resumir(baseline), **{n: h.resumir(g) for n, g in filas.groupby('sistema', sort=False)}}).T
display(resumen.reindex(columns=columnas_resumen).round(3))

### ¿Llega la norma esperada?

`recall_3` por caso y por sistema: fracción de las normas citadas en `expected` que está entre las tres referencias. Es la
lectura más directa de qué técnica mejora la recuperación. Los casos de premisa falsa y fuera de dominio no citan norma.

In [ ]:
recall = filas.pivot(index='id', columns='sistema', values='recall_3')[list(sistemas)].dropna(how='all')
display(recall.round(2))
print('Casos con al menos una norma esperada en el top-3:')
print((recall > 0).sum().to_string())

### Diagnóstico del RAG ingenuo, caso por caso

Tipo de falla de cada caso en el sistema A: qué parte de las normas esperadas llegó al top-3.

In [ ]:
def tipo_falla(caso, recall_caso):
    if caso['case_type'] in h.CASOS_ABSTENCION:
        return 'sin norma esperada: el resumen debe reconocer que las referencias no resuelven la consulta'
    if recall_caso == 0:
        return 'la norma esperada no llegó al top-3'
    if recall_caso < 1:
        return 'llegó solo parte de las normas esperadas'
    return 'la norma esperada llegó al top-3'

a = filas[filas['sistema'] == 'A_ingenuo'].set_index('id')
casos = {e['id']: e for e in eval_set}
diagnostico = pd.DataFrame({'tipo': [tipo_falla(casos[i], a.loc[i, 'recall_3']) for i in a.index],
                            'acierto_A': a['acierto_dominio']}, index=a.index)
display(diagnostico)
print(diagnostico['tipo'].value_counts().to_string())

### Control: el sistema A reproduce al ingenuo

El sistema A usa la misma recuperación que `rag_ingenuo.ipynb`; debe traer los mismos artículos.

In [ ]:
ingenuo = pd.read_csv(RAIZ / 'notebooks/rag/resultados/ingenuo/scorecard_rag_ingenuo_detalle.csv').set_index('id')
det_a = detalle[detalle['sistema'] == 'A_ingenuo'].set_index('id')
iguales = sum(det_a.loc[i, 'articulos_recuperados'] == ingenuo.loc[i, 'articulos_recuperados'] for i in det_a.index)
print(f'A recupera los mismos artículos que el RAG ingenuo en {iguales}/{len(det_a)} casos.')

## 8. Exportación

`scorecard_rag_avanzado.csv` (mismas columnas que el baseline y el ingenuo) y su detalle corresponden a `SISTEMA_AVANZADO`.
Los archivos de ablación traen los cinco sistemas.

In [ ]:
from pathlib import Path

h.escribir_scorecards(Path('.'), 'rag_avanzado', resultados_sistemas[SISTEMA_AVANZADO])
filas.to_csv('ablacion_rag_avanzado.csv', index=False, lineterminator='\n')
detalle.to_csv('ablacion_rag_avanzado_detalle.csv', index=False, lineterminator='\n')

archivos = ['scorecard_rag_avanzado.csv', 'scorecard_rag_avanzado_detalle.csv',
            'ablacion_rag_avanzado.csv', 'ablacion_rag_avanzado_detalle.csv']
if EN_COLAB:
    from google.colab import files
    for archivo in archivos:
        files.download(archivo)
else:
    print('Guardados en el directorio actual:', ', '.join(archivos))

## 9. Interpretación y limitaciones

Se completa con los resultados de la corrida.